In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'SAS': ['Emanuel Miller'], 'DEN': ['Zeke Nnaji']}

Out Players:
{'ATL': ['Jock Landale'], 'DET': ['Caris LeVert', 'Isaiah Stewart', 'Cade Cunningham', 'Tobias Harris', 'Duncan Robinson'], 'ORL': ['Jonathan Isaac', 'Franz Wagner', 'Jett Howard'], 'PHI': ['Johni Broome', 'Cameron Payne'], 'CLE': ['Donovan Mitchell', 'Jaylon Tyson', 'Thomas Bryant', 'Dean Wade', 'James Harden', 'Max Strus'], 'MEM': ['Javon Small', 'Jahmai Mashack', 'Ty Jerome'], 'POR': ['Shaedon Sharpe', 'Vít Krejčí', 'Jerami Grant'], 'DEN': ['Spencer Jones', 'Peyton Watson']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 10 teams with confirmed lineups
Updated 2 teams with questionable players


### Dataset

In [12]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
150,NaN,2025-26,1643253,Toby Okani,Toby,1610612763,MEM,Memphis Grizzlies,22501135,2026-04-05T00:00:00,MEM @ MIL,L,41.166667,4,12,0.333,1,4,0.25,0,2,0.000,2,1,3,0,1,1,0,1,4,2,9,-11,14.6,0,0,15.0,1,41:10,1,113.5,113.5,113.5,124.9,127.3,127.3,-11.4,-13.8,-13.8,0.000,0.0,0.0,0.041,0.029,0.036,7.1,7.2,0.375,0.349,0.133,0.136,104.17,103.19,85.99,103.19,-0.016,89,4.0,12.0,38,92,0.413,17,46,0.370,22,28,0.786,15,23,38,25,14.0,13,1,5,21,19,115,-16.0,111.3,112.7,127.7,129.7,-16.4,-17.0,0.658,1.79,17.2,0.321,0.649,0.452,0.137,0.505,0.551,102.9,101.5,84.58,102,0.403,1610612749,MIL,Milwaukee Bucks,50,83,0.602,16,32,0.500,15,24,0.625,11,33,44,30,20.0,11,5,1,19,21,131,16.0,127.7,129.7,111.3,112.7,16.4,17.0,0.600,1.50,20.8,0.351,0.679,0.548,0.198,0.699,0.700,102.9,101.5,84.58,101,0.597,G,NaN,NaN
149,NaN,2025-26,1629001,De'Anthony Melton,De'Anthony,1610612744,GSW,Golden State Warriors,22501142,2026-04-05T00:00:00,GSW vs. HOU,L,22.840000,2,4,0.500,2,4,0.50,0,0,0.000,0,1,1,4,1,0,1,0,0,0,6,-7,15.2,0,0,15.0,1,22:50,1,132.0,127.3,127.3,142.9,140.0,140.0,-11.0,-12.7,-12.7,0.200,4.0,44.4,0.000,0.083,0.031,11.1,11.1,0.750,0.750,0.100,0.108,90.91,93.52,77.93,93.52,0.064,44,2.0,4.0,42,84,0.500,14,40,0.350,18,20,0.900,6,25,31,34,10.0,7,3,7,19,16,116,-1.0,119.8,123.4,124.0,124.5,-4.2,-1.1,0.810,3.40,24.5,0.256,0.737,0.481,0.106,0.583,0.625,95.6,94.0,78.33,94,0.487,1610612745,HOU,Houston Rockets,44,80,0.550,13,29,0.448,16,19,0.842,8,30,38,30,14.0,4,7,3,16,19,117,1.0,124.0,124.5,119.8,123.4,4.2,1.1,0.682,2.14,22.4,0.263,0.744,0.519,0.149,0.631,0.662,95.6,94.0,78.33,94,0.513,G,PG,27.0
148,NaN,2025-26,1631288,Jamal Cain,Jamal,1610612753,ORL,Orlando Magic,22501138,2026-04-05T00:00:00,ORL @ NOP,W,27.621667,3,7,0.429,0,3,0.00,2,2,1.000,2,3,5,2,0,0,0,0,2,4,8,11,17.0,0,0,15.0,1,27:37,1,123.6,127.8,127.8,99.7,101.8,101.8,23.8,26.0,26.0,0.087,0.0,20.0,0.067,0.111,0.088,0.0,0.0,0.429,0.508,0.119,0.118,99.05,96.45,80.37,96.45,0.068,54,3.0,7.0,40,93,0.430,7,33,0.212,25,36,0.694,16,41,57,27,13.0,6,5,8,23,27,112,4.0,105.8,107.7,102.0,102.9,3.8,4.8,0.675,2.08,17.9,0.345,0.778,0.554,0.125,0.468,0.515,105.8,104.5,87.08,104,0.545,1610612740,NOP,New Orleans Pelicans,35,84,0.417,10,31,0.323,28,36,0.778,9,38,47,18,15.0,7,8,5,27,23,108,-4.0,102.0,102.9,105.8,107.7,-3.8,-4.8,0.514,1.20,13.4,0.222,0.655,0.446,0.143,0.476,0.541,105.8,104.5,87.08,105,0.455,NaN,SF,26.0
160,NaN,2025-26,1630644,Mac Mc

### Load latest odds on file

In [13]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260406_002052.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,New York Knicks,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Orlando Magic,Detroit Pistons,2026-04-06 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Memphis Grizzlies,Cleveland Cavaliers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,San Antonio Spurs,Philadelphia 76ers,2026-04-07 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Denver Nuggets,Portland Trail Blazers,2026-04-07 01:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [25]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
BOOKMAKER = 'Underdog'
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['BOOKMAKER'] == BOOKMAKER) & (lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-06 00:20:52
US latest pull: 2026-04-06 00:19:59


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
250,Underdog,player_points,Karl-Anthony Towns,Over,18.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
251,Underdog,player_points,Karl-Anthony Towns,Under,18.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
252,Underdog,player_points,Jalen Brunson,Over,24.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
253,Underdog,player_points,Jalen Brunson,Under,24.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52
254,Underdog,player_points,Jonathan Kuminga,Over,10.5,-137,2026-04-06,2026-04-06T07:20:00Z,2026-04-06 00:20:52


### Load my models

In [26]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [27]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] G.G. Jackson: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90
0,Jalen Johnson,AST,28.62,34.20,40.26,0.0705,0.1685,0.2632,2.02,5.76,10.60
1,Nickeil Alexander-Walker,AST,20.10,29.08,36.65,0.0385,0.1110,0.1910,0.77,3.23,7.00
2,Desmond Bane,AST,27.43,32.47,38.75,0.0500,0.1264,0.2330,1.37,4.10,9.03
3,Evan Mobley,AST,23.12,30.32,37.68,0.0563,0.1098,0.2064,1.30,3.33,7.78
4,Cedric Coward,AST,17.94,26.05,32.16,0.0338,0.1130,0.2023,0.61,2.94,6.51
5,De'Aaron Fox,AST,22.09,30.60,38.75,0.1122,0.1790,0.3267,2.48,5.48,12.66
6,VJ Edgecombe,AST,26.51,33.68,39.33,0.0415,0.1277,0.2291,1.10,4.30,9.01


### Get Line Probabilities

In [28]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
51,Quentin Grimes,PTS,8.5,26.53,10.84,0.843,0.157
58,Cameron Johnson,PTS,12.5,28.92,13.46,0.659,0.341
42,Victor Wembanyama,PTS,28.5,30.21,23.98,0.313,0.687
11,James Harden,REB,4.5,33.22,4.56,0.625,0.374
25,Josh Hart,PTS,12.5,30.70,10.01,0.379,0.621
38,James Harden,PTS,19.5,33.22,20.16,0.602,0.398
53,Nikola Jokić,PTS,27.5,34.88,23.34,0.458,0.542
24,OG Anunoby,PTS,15.5,34.49,17.70,0.684,0.316
60,Tim Hardaway Jr.,PTS,11.5,25.78,11.96,0.655,0.345
34,Jalen Duren,PTS,22.5,29.33,16.80,0.271,0.729


In [ ]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker=BOOKMAKER,
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
12,VJ Edgecombe,AST,3.5,33.68,4.30,0.666,0.334,AST,PrizePicks,San Antonio Spurs,8.5,236.5,110.2,3.0,100.77,12.0,-115.0,-104.0,0.535,0.510,5.1,4.0,3.14,1.6,0.5,-0.510,0.695,0.305,29.93,-40.17,0.4,0.7,0.60,0.58,33.52,5.41,0.22,0.06,1.00,1.0
33,Tyrese Maxey,REB,4.0,35.51,3.68,0.523,0.344,REB,PrizePicks,San Antonio Spurs,8.5,236.5,110.2,3.0,100.77,12.0,-137.0,-137.0,0.578,0.578,4.2,3.5,2.57,0.2,-0.5,-0.078,0.531,0.469,-8.14,-18.87,0.6,0.4,0.33,0.29,37.10,5.74,0.29,0.05,9.00,2.0
65,Tristan da Silva,PTS,8.5,26.98,10.28,0.737,0.263,PTS,PrizePicks,Detroit Pistons,3.0,225.0,108.6,2.0,99.91,19.0,-122.0,100.0,0.550,0.500,12.4,12.0,6.52,3.9,3.5,-0.598,0.725,0.275,31.93,-45.00,0.8,0.8,0.80,0.44,26.51,6.02,0.16,0.05,9.17,6.0
60,Jalen Suggs,PTS,14.5,27.95,13.00,0.558,0.442,PTS,PrizePicks,Detroit Pistons,3.0,225.0,108.6,2.0,99.91,19.0,-116.0,-110.0,0.537,0.524,12.3,12.0,4.57,-2.2,-2.5,0.481,0.315,0.685,-41.34,30.77,0.4,0.2,0.27,0.41,30.57,4.87,0.21,0.04,11.67,6.0
77,Stephon Castle,PTS,17.5,29.92,17.81,0.549,0.451,PTS,PrizePicks,Philadelphia 76ers,-8.5,236.5,114.9,17.0,100.32,16.0,-137.0,-137.0,0.578,0.578,17.3,19.5,5.83,-0.2,2.0,0.034,0.486,0.514,-15.93,-11.08,0.8,0.6,0.60,0.41,30.63,6.00,0.23,0.04,16.33,3.0


In [29]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker=BOOKMAKER,
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
19,Donovan Clingan,REB,11.5,25.89,10.89,0.466,0.534,REB,Underdog,Denver Nuggets,8.5,240.5,116.0,21.0,99.47,20.0,-119.0,-102.0,0.543,0.505,11.4,12.5,4.27,-0.1,1.0,0.023,0.491,0.509,-9.64,0.80,0.4,0.6,0.60,0.36,25.84,3.61,0.19,0.07,11.00,6.0
8,Dyson Daniels,REB,6.5,30.39,5.34,0.452,0.548,REB,Underdog,New York Knicks,-1.5,229.0,112.3,8.0,97.97,25.0,-105.0,-114.0,0.512,0.533,7.4,6.0,3.81,0.9,-0.5,-0.236,0.593,0.407,15.78,-23.60,0.4,0.4,0.53,0.43,32.25,5.62,0.16,0.03,5.14,7.0
44,De'Aaron Fox,PTS,15.5,30.60,20.11,0.806,0.194,PTS,Underdog,Philadelphia 76ers,-8.5,236.5,114.9,17.0,100.32,16.0,-127.0,-101.0,0.559,0.502,15.0,14.0,4.88,-0.5,-1.5,0.102,0.459,0.541,-17.96,7.66,0.2,0.3,0.53,0.68,27.70,5.89,0.23,0.05,21.67,3.0
4,Cedric Coward,AST,2.5,26.05,2.94,0.608,0.392,AST,Underdog,Cleveland Cavaliers,13.5,238.5,114.0,14.0,100.60,13.0,-130.0,100.0,0.565,0.500,2.5,2.5,1.58,0.0,0.0,0.000,0.500,0.500,-11.54,0.00,0.4,0.5,0.47,0.53,25.24,1.62,0.22,0.07,2.00,1.0
41,Cedric Coward,PTS,13.5,26.05,14.37,0.640,0.359,PTS,Underdog,Cleveland Cavaliers,13.5,238.5,114.0,14.0,100.60,13.0,-112.0,-115.0,0.528,0.535,13.5,14.0,5.04,0.0,0.5,0.000,0.500,0.500,-5.36,-6.52,0.6,0.5,0.47,0.45,25.24,1.62,0.22,0.07,10.00,1.0


In [31]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
16,Bruce Brown,AST,1.5,23.58,1.98,0.724,0.276,AST,PrizePicks,Portland Trail Blazers,-8.5,240.5,113.5,13.0,101.85,7.0,-137.0,-137.0,0.578,0.578,1.4,1.0,1.07,-0.1,-0.5,0.093,0.463,0.537,-19.90,-7.10,0.0,0.4,0.47,0.56,20.70,4.90,0.17,0.05,1.33,3.0
87,Jamal Murray,PTS,24.5,34.14,22.54,0.457,0.543,PTS,PrizePicks,Portland Trail Blazers,-8.5,240.5,113.5,13.0,101.85,7.0,-122.0,-102.0,0.550,0.505,26.1,21.5,12.22,1.6,-3.0,-0.131,0.552,0.448,0.45,-11.28,0.6,0.4,0.40,0.39,37.33,5.70,0.24,0.06,25.00,7.0
81,Keldon Johnson,PTS,12.5,26.70,17.05,0.767,0.233,PTS,PrizePicks,Philadelphia 76ers,-8.5,236.5,114.9,17.0,100.32,16.0,-104.0,-114.0,0.510,0.533,14.9,15.0,5.30,2.4,2.5,-0.453,0.675,0.325,32.40,-38.99,0.6,0.7,0.60,0.46,23.09,3.15,0.22,0.05,11.33,3.0
67,Donovan Mitchell,PTS,25.5,31.00,23.44,0.512,0.488,PTS,PrizePicks,Memphis Grizzlies,-13.5,238.5,117.8,24.0,101.43,9.0,100.0,-116.0,0.500,0.537,25.5,26.5,11.41,0.0,1.0,0.000,0.500,0.500,0.00,-6.90,0.4,0.6,0.47,0.53,33.77,4.28,0.29,0.07,31.50,2.0
24,OG Anunoby,PTS,15.5,34.49,17.70,0.684,0.316,PTS,Underdog,Atlanta Hawks,1.5,229.0,112.7,9.0,102.54,5.0,-115.0,-102.0,0.535,0.505,17.7,16.5,7.86,2.2,1.0,-0.280,0.610,0.390,14.04,-22.76,0.6,0.6,0.67,0.57,34.29,4.58,0.18,0.04,15.67,6.0


### Get top EVs

In [30]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 55  |  Pairs: 126  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
